# Generate public-safe dissertation figures (aggregate)

This notebook regenerates **public-safe aggregate** dissertation figures from the cleaned GitHub repository.

## Purpose
- Produce presentation-quality **aggregate** figures for Figure 2 and Figure 3 (confusion matrix).
- Avoid any use of protected participant-level audio, annotations, or clinical labels.

## Required inputs
- `results/final_matched_metrics_table.csv`

## Privacy note
Raw clinical audio, participant-level labels, and private annotation exports are excluded from this repository.

If you have access to the protected dataset, those figures can be regenerated locally in a secure environment.


In [ ]:
from __future__ import annotations

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

sns.set_context("talk")
sns.set_style("whitegrid")


def find_repo_root(start: Path) -> Path:
    """Find the repository root by walking upward until `results/` is found."""

    start = start.resolve()
    for p in [start] + list(start.parents):
        if (p / "results" / "final_matched_metrics_table.csv").exists():
            return p
    return start


# nbconvert execution may set the working directory to the notebook folder.
# We locate the repo root by searching upward for the expected results file.
REPO_ROOT = find_repo_root(Path.cwd())
RESULTS_CSV = REPO_ROOT / "results" / "final_matched_metrics_table.csv"
OUT_DIR = REPO_ROOT / "figures" / "main" / "regenerated"
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("CWD:", Path.cwd().resolve())
print("Repo root:", REPO_ROOT)
print("Using:", RESULTS_CSV)
print("Writing to:", OUT_DIR)


## Load aggregate metrics table

This table contains **aggregate** coverage and AF/SR-classified diagnostic metrics for each method on the matched-to-PCG cohort (`n_ref=67`).


In [ ]:
df = pd.read_csv(RESULTS_CSV)

display(df)

assert set(df["n_ref"].tolist()) == {67}, "Expected matched cohort denominator n_ref=67 for all methods"


## Figure 2 (regenerated): Coverage and AF/SR-classified performance

Panel A uses the matched cohort denominator (`n_ref=67`) and shows the proportion of outputs that are:
- AF/SR-classified
- OA (other arrhythmia / indeterminate)
- UI (uninterpretable)
- Missing

Panel B reports sensitivity, specificity, and accuracy **computed only among AF/SR-classified outputs**.


In [ ]:
order = ["PCG", "FibriCheck iOS", "FibriCheck Android", "Kardia"]
df2 = df.set_index("method").loc[order].reset_index()

# Coverage proportions
coverage = pd.DataFrame({
    "method": df2["method"],
    "AF/SR-classified": df2["classified_rate"],
    "OA": df2["OA_rate"],
    "UI": df2["UI_rate"],
    "Missing": df2["missing_rate"],
}).set_index("method")

# Performance metrics on AF/SR-classified subset
perf = pd.DataFrame({
    "method": df2["method"],
    "Sensitivity": df2["sensitivity"],
    "Specificity": df2["specificity"],
    "Accuracy": df2["accuracy"],
}).set_index("method")

coverage, perf


In [ ]:
# Figure 2 (publication-style): coverage and AF/SR-classified performance
#
# Notes
# -----
# - This figure uses *locked* counts/metrics for the matched cohort (n_ref=67),
#   and exists as a privacy-preserving, fully reproducible aggregate figure.
# - X-axis tick labels are device names only (no "n=..." text).

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Locked cohort and locked counts/metrics (matched-to-PCG, n_ref=67).
n_ref = 67
methods = ["PCG", "Fibri iOS", "Fibri Android", "Kardia"]

# Coverage components: AF/SR-classified / OA / UI / Missing
classified_n = np.array([26, 61, 38, 57], dtype=int)
oa_n = np.array([0, 4, 2, 7], dtype=int)
ui_n = np.array([41, 2, 27, 1], dtype=int)
missing_n = np.array([0, 0, 0, 2], dtype=int)

classified = classified_n / n_ref
oa = oa_n / n_ref
ui = ui_n / n_ref
missing = missing_n / n_ref

# AF/SR-classified performance (computed only among AF/SR-classified outputs)
sens = np.array([0.875, 1.0, 1.0, 1.0], dtype=float)
spec = np.array([0.8888888889, 0.9777777778, 1.0, 0.9772727273], dtype=float)
acc = np.array([0.8846153846, 0.9836065574, 1.0, 0.9824561404], dtype=float)

# Match the prior publication-style baseline (from the original analysis export)
# as closely as possible.
sns.set_theme(style="whitegrid")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15.8, 6.3))
fig.suptitle(
    "Coverage and AF/SR-classified performance vs ECG12 AF/SR reference",
    fontsize=20,
    fontweight="bold",
    y=0.98,
)

x = np.arange(len(methods))

# Panel A: coverage (stacked)
ax1.set_title("A. Coverage", fontsize=18, pad=16)
ax1.bar(x, classified, label="Classified AF/SR", color="#f6c667", edgecolor="white", linewidth=1.0)
ax1.bar(x, oa, bottom=classified, label="OA", color="#9ecae1", edgecolor="white", linewidth=1.0)
ax1.bar(x, ui, bottom=classified + oa, label="UI", color="#f28e8e", edgecolor="white", linewidth=1.0)
ax1.bar(x, missing, bottom=classified + oa + ui, label="Missing", color="#bdbdbd", edgecolor="white", linewidth=1.0)

ax1.set_ylim(0, 1.0)
ax1.set_xticks(x)
ax1.set_xticklabels(methods, fontsize=13)
ax1.tick_params(axis="x", pad=14)
ax1.set_ylabel("Proportion of cohort (ECG12 AF/SR with PCG available)", fontsize=15)

# Keep the original bar annotations inside Panel A.
for i, n in enumerate(classified_n):
    pct = (n / n_ref) * 100
    y = max(0.06, classified[i] / 2)
    ax1.text(
        i,
        y,
        f"{n}/{n_ref}\nclassified\n({pct:.1f}%)",
        ha="center",
        va="center",
        fontsize=13,
        color="black",
    )

ax1.legend(loc="upper right", frameon=True, fontsize=14)

# Cohort footnote
fig.text(0.08, 0.06, f"Cohort: ECG12 AF/SR with PCG available (n={n_ref})", fontsize=14, color="#555555")

# Panel B: performance (grouped)
ax2.set_title("B. AF/SR-classified performance", fontsize=18, pad=16)
width = 0.24
ax2.bar(x - width, sens, width, label="Sensitivity", color="#4c78a8", edgecolor="white", linewidth=0.8)
ax2.bar(x, spec, width, label="Specificity", color="#59a14f", edgecolor="white", linewidth=0.8)
ax2.bar(x + width, acc, width, label="Accuracy", color="#f28e2b", edgecolor="white", linewidth=0.8)

ax2.set_ylim(0, 1.0)
ax2.set_xticks(x)
ax2.set_xticklabels(methods, fontsize=13)
ax2.tick_params(axis="x", pad=14)
ax2.set_ylabel("Metric value", fontsize=15)
ax2.legend(loc="lower right", frameon=True, fontsize=14)

plt.subplots_adjust(top=0.80, bottom=0.18, wspace=0.22)

out_png = REPO_ROOT / "figures" / "main" / "regenerated" / "fig2_coverage_performance.png"
out_pdf = REPO_ROOT / "figures" / "main" / "regenerated" / "fig2_coverage_performance.pdf"
out_png.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(out_png, bbox_inches="tight", dpi=300)
fig.savefig(out_pdf, bbox_inches="tight")
plt.close(fig)

print("Wrote:", out_png)
print("Wrote:", out_pdf)


## Figure 3 (regenerated): PCG confusion matrix (aggregate counts)

The public repository does **not** include participant-level probabilities, so the **ROC curve cannot be regenerated** exactly from aggregate tables alone.

However, the confusion matrix for the AF/SR-classified subset can be shown using aggregate counts.


In [ ]:
# Aggregate confusion matrix counts for PCG on the AF/SR-classified subset (public-safe)
# Rows: ECG12 reference (SR, AF); Columns: PCG prediction (SR, AF)
#
# True SR predicted SR: 16
# True SR predicted AF: 2
# True AF predicted SR: 1
# True AF predicted AF: 7
cm = np.array([[16, 2], [1, 7]], dtype=int)
cm


In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))

sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    cbar=False,
    linewidths=0.0,
    ax=ax,
)

ax.set_xlabel("Predicted label")
ax.set_ylabel("ECG12 reference")
ax.set_xticklabels(["SR", "AF"])
ax.set_yticklabels(["SR", "AF"], rotation=0)
ax.set_title("PCG confusion matrix (AF/SR-classified subset)")

out_png = OUT_DIR / "fig3_pcg_confusion_matrix_regenerated.png"
out_pdf = OUT_DIR / "fig3_pcg_confusion_matrix_regenerated.pdf"
fig.savefig(out_png, dpi=300, bbox_inches="tight")
fig.savefig(out_pdf, bbox_inches="tight")
plt.close(fig)

print("Wrote:", out_png)
print("Wrote:", out_pdf)
